In [ ]:
# =========================================================
# Load Libraries
# =========================================================
import os
import time
# Run from the notebooks/ directory OR set kernel working directory to HAB_Phase1/
import sys, pathlib

import itertools

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from tensorflow import keras

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

# =========================================================
# Paths
# =========================================================

PROJECT_ROOT = pathlib.Path().resolve()
while PROJECT_ROOT.name != "HAB_Phase1":
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not find HAB_Phase1 project root")

sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_PATH, OUTPUT_DIRS

# DATA_PATH  = "Data/UCDavis_Data_for_Analysis2_220days_ToxinProdMicrocystis.csv"
# OUTPUT_DIR = "C:/Temp/FC_ANN_Results"

OUTPUT_DIR = str(OUTPUT_DIRS["02"])  # change for each notebook
DATA_PATH  = str(DATA_PATH)


def setup_output_dirs():
    subdirs = [
        "figures/model_comparison",
        "figures/confusion_matrix",
        "figures/dropout_impact",
        "tables",
        "logs",
        "models/best_models",
        "models/best_configurations",
    ]
    for subdir in subdirs:
        os.makedirs(os.path.join(OUTPUT_DIR, subdir), exist_ok=True)


setup_output_dirs()

# Convenience path variables
TABLES_DIR      = os.path.join(OUTPUT_DIR, "tables")
DROPOUT_FIG_DIR = os.path.join(OUTPUT_DIR, "figures", "dropout_impact")
BEST_MODELS_DIR = os.path.join(OUTPUT_DIR, "models", "best_models")
CONFIGS_DIR     = os.path.join(OUTPUT_DIR, "models", "best_configurations")

# =========================================================
# Result CSV File
# =========================================================
result_file = os.path.join(
    TABLES_DIR,
    "ANN_results_with_hyperparams_5_feature_Combinations.csv",
)

if not os.path.exists(result_file):
    pd.DataFrame(columns=[
        "Feature Combination Index",
        "Model",
        "Features",
        "Best Parameters",
        "Train Accuracy",
        "Test Accuracy",
        "Train Precision",
        "Test Precision",
        "Train Recall",
        "Test Recall",
        "Train F1 Score",
        "Test F1 Score",
    ]).to_csv(result_file, index=False)

# =========================================================
# Load Dataset
# =========================================================
df = pd.read_csv(DATA_PATH)

TARGET_COLUMN = "ToxinProdc_Microcystis_CC"
THRESHOLD     = 4000

# =========================================================
# Feature Columns  (17 features — single source of truth)
# =========================================================
feature_columns = [
    "Q", "Q_2w", "Q_1m",
    "SpecCond.", "Salinity", "NTU",
    "OrgC", "Total_OrgC",
    "NH4", "NO3", "OrgN",
    "PO4", "TP",
    "WaterTemp", "DO", "pH", "TDS",
]

# Verify every feature is present in the CSV
missing_cols = [f for f in feature_columns if f not in df.columns]
if missing_cols:
    raise ValueError(
        f"The following features are not found in the CSV:\n  {missing_cols}\n"
        f"Available columns: {df.columns.tolist()}"
    )

print(f"Feature columns ({len(feature_columns)}): {feature_columns}")

# =========================================================
# Build X and y
# =========================================================
X = df[feature_columns].copy()
y = np.where(df[TARGET_COLUMN] <= THRESHOLD, 0, 1)

print(f"\nDataset shape  : X={X.shape}, y={y.shape}")
print(f"Class distribution: 0={np.sum(y == 0)}, 1={np.sum(y == 1)}")

# =========================================================
# 70 / 30 Stratified Train-Test Split
# =========================================================
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print(f"\nTrain size : {X_train_raw.shape[0]} samples")
print(f"Test  size : {X_test_raw.shape[0]}  samples")
print(f"Train class dist: 0={np.sum(y_train == 0)}, 1={np.sum(y_train == 1)}")
print(f"Test  class dist: 0={np.sum(y_test  == 0)}, 1={np.sum(y_test  == 1)}")

# =========================================================
# Fit Fresh StandardScaler on Training Data Only
# =========================================================
scaler = StandardScaler()
scaler.fit(X_train_raw)          # fit on train only

# Full scaled arrays (all 17 features); columns sliced per combination later
X_train_full_scaled = scaler.transform(X_train_raw)
X_test_full_scaled  = scaler.transform(X_test_raw)

print(f"\nFresh StandardScaler fitted on {X_train_raw.shape[0]} training samples.")

# =========================================================
# Feature Combinations
# =========================================================
feature_combinations = [
    ['Q', 'Q_1m', 'SpecCond.', 'OrgC', 'OrgN', 'PO4',
     'WaterTemp', 'DO', 'pH'],

    ['Q', 'SpecCond.', 'OrgC', 'Total_OrgC', 'NH4',
     'OrgN', 'PO4', 'WaterTemp', 'TDS'],

    ['Q_2w', 'Q_1m', 'SpecCond.', 'OrgC', 'OrgN', 'PO4', 
     'WaterTemp', 'DO', 'pH', 'TDS'],

    ['Q', 'Q_2w', 'SpecCond.', 'Salinity', 'OrgC',
     'Total_OrgC', 'OrgN', 'PO4', 'WaterTemp',
     'DO', 'pH', 'TDS'],

    ['Q', 'Q_2w', 'Q_1m', 'SpecCond.', 'Salinity',
     'NTU', 'OrgC', 'Total_OrgC', 'NH4', 'NO3',
     'OrgN', 'PO4', 'TP', 'WaterTemp', 'DO',
     'pH', 'TDS'],
]

# =========================================================
# VALIDATE FEATURE COMBINATIONS AGAINST feature_columns
# =========================================================
print("\nValidating feature combinations against the 17-feature column list...")

for combo_idx, combo in enumerate(feature_combinations, start=1):
    missing = [f for f in combo if f not in feature_columns]
    if missing:
        raise ValueError(
            f"Feature combination {combo_idx} contains features not found "
            f"in the 17-feature list:\n"
            f"  Missing:     {missing}\n"
            f"  Combination: {combo}\n"
            f"  Available:   {feature_columns}"
        )

print(f"All {len(feature_combinations)} feature combinations validated successfully.")

# =========================================================
# ANN Parameter Grid
# =========================================================
param_grid = {
    'hidden_layers': [2, 3, 4],
    'activation':    ['relu', 'elu', 'sigmoid', 'tanh'],
    'neurons':       [16, 32, 64, 128],
    'learning_rate': [0.0001, 0.0006],
    'dropout_rate':  [0.1, 0.15, 0.2, 0.25, 0.3],
}

# =========================================================
# Parameter Combination Limit
#   None   -> Run ALL combinations in the full grid
#   Integer -> Run only that many combinations (e.g. 10)
# =========================================================
n_param_combinations = None  # set to None for full run

if n_param_combinations is None:
    all_params = list(ParameterGrid(param_grid))
    print(f"\nRunning ALL {len(all_params)} parameter combinations")
else:
    all_params = list(itertools.islice(ParameterGrid(param_grid), n_param_combinations))
    print(f"\nRunning first {len(all_params)} of "
          f"{len(list(ParameterGrid(param_grid)))} parameter combinations")

# =========================================================
# Process Feature Combinations
# =========================================================
for combination_index, feature_combination in enumerate(feature_combinations, start=1):

    print("\n===================================================")
    print(f"Processing Feature Combination {combination_index}")
    print(f"Features: {feature_combination}")
    print("===================================================")

    start_time = time.time()

    # -------------------------------------------------------
    # Slice Scaled Arrays to this combination's columns
    # -------------------------------------------------------
    feature_indices = [feature_columns.index(f) for f in feature_combination]

    X_train_scaled = X_train_full_scaled[:, feature_indices]
    X_test_scaled  = X_test_full_scaled[:,  feature_indices]

    # -------------------------------------------------------
    # Initialize Best Tracking
    # -------------------------------------------------------
    best_score  = -1
    best_model  = None
    best_config = None

    dropout_impact     = {}
    total_combinations = len(all_params)

    # =====================================================
    # Hyperparameter Search
    # =====================================================
    for idx, params in enumerate(all_params, start=1):

        print(f"\nTraining ANN Combination {idx}/{total_combinations}")
        print(
            f"  Layers: {params['hidden_layers']}, "
            f"Activation: {params['activation']}, "
            f"Neurons: {params['neurons']}, "
            f"Learning Rate: {params['learning_rate']}, "
            f"Dropout Rate: {params['dropout_rate']}"
        )

        # -------------------------------------------------
        # Build ANN Model
        # -------------------------------------------------
        model = keras.Sequential()

        model.add(keras.layers.Input(shape=(X_train_scaled.shape[1],)))

        # First hidden layer
        model.add(keras.layers.Dense(params['neurons'], activation=params['activation']))
        model.add(keras.layers.Dropout(params['dropout_rate']))

        # Additional hidden layers
        for _ in range(params['hidden_layers'] - 1):
            model.add(keras.layers.Dense(params['neurons'], activation=params['activation']))
            model.add(keras.layers.Dropout(params['dropout_rate']))

        # Output layer
        model.add(keras.layers.Dense(1, activation='sigmoid'))

        # -------------------------------------------------
        # Compile
        # -------------------------------------------------
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=params['learning_rate']),
            loss='binary_crossentropy',
            metrics=['accuracy'],
        )

        # -------------------------------------------------
        # Early Stopping
        # -------------------------------------------------
        early_stopping = keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=30,
            restore_best_weights=True,
        )

        # -------------------------------------------------
        # Train
        # -------------------------------------------------
        history = model.fit(        # noqa: F841  (history kept for potential future use)
            X_train_scaled,
            y_train,
            epochs=10,
            batch_size=32,
            validation_split=0.2,
            callbacks=[early_stopping],
            verbose=0,
        )

        # -------------------------------------------------
        # Predictions
        # -------------------------------------------------
        y_train_pred = (model.predict(X_train_scaled, verbose=0) >= 0.5).astype(int)
        y_test_pred  = (model.predict(X_test_scaled,  verbose=0) >= 0.5).astype(int)

        # -------------------------------------------------
        # Metrics
        # -------------------------------------------------
        train_accuracy  = accuracy_score(y_train,  y_train_pred)
        test_accuracy   = accuracy_score(y_test,   y_test_pred)
        train_precision = precision_score(y_train, y_train_pred, zero_division=0)
        test_precision  = precision_score(y_test,  y_test_pred,  zero_division=0)
        train_recall    = recall_score(y_train,    y_train_pred, zero_division=0)
        test_recall     = recall_score(y_test,     y_test_pred,  zero_division=0)
        train_f1        = f1_score(y_train,        y_train_pred, zero_division=0)
        test_f1         = f1_score(y_test,         y_test_pred,  zero_division=0)

        # -------------------------------------------------
        # Track Dropout Impact
        # -------------------------------------------------
        dr = params['dropout_rate']
        if dr not in dropout_impact:
            dropout_impact[dr] = []
        dropout_impact[dr].append(test_accuracy)

        # -------------------------------------------------
        # Update Best Model
        # -------------------------------------------------
        if test_accuracy > best_score:

            best_score  = test_accuracy
            best_model  = model
            best_config = {
                "Feature Combination Index": combination_index,
                "Model":           "ANN",
                "Features":        feature_combination,
                "Best Parameters": params,
                "Train Accuracy":  train_accuracy,
                "Test Accuracy":   test_accuracy,
                "Train Precision": train_precision,
                "Test Precision":  test_precision,
                "Train Recall":    train_recall,
                "Test Recall":     test_recall,
                "Train F1 Score":  train_f1,
                "Test F1 Score":   test_f1,
            }

            best_model.save(
                os.path.join(
                    BEST_MODELS_DIR,
                    f"best_model_combination_{combination_index}.h5",
                )
            )

    # =====================================================
    # Save Best Result to CSV
    # =====================================================
    pd.DataFrame([best_config]).to_csv(
        result_file,
        mode='a',
        header=False,
        index=False,
    )

    # =====================================================
    # Save Best Config PKL
    # (includes scaler so downstream inference can reproduce
    #  the same transform without re-fitting)
    # =====================================================
    joblib.dump(
        {
            **best_config,
            "scaler":          scaler,
            "scaler_features": feature_columns,
            "feature_indices": feature_indices,
        },
        os.path.join(CONFIGS_DIR, f"best_model_config_combination_{combination_index}.pkl"),
    )

    # =====================================================
    # Plot Dropout Impact
    # =====================================================
    plt.figure(figsize=(10, 6))

    for dr, accuracies in sorted(dropout_impact.items()):
        plt.plot(range(1, len(accuracies) + 1), accuracies, label=f"Dropout Rate {dr}")

    plt.xlabel("Configurations")
    plt.ylabel("Test Accuracy")
    plt.title(f"Dropout Impact — Feature Combination {combination_index}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            DROPOUT_FIG_DIR,
            f"dropout_impact_combination_{combination_index}.png",
        )
    )
    plt.close()

    # =====================================================
    # Time Reporting
    # =====================================================
    elapsed_time = time.time() - start_time
    print(f"\nFeature Combination {combination_index} completed in {elapsed_time:.2f} seconds.")
    print(f"  Best Test Accuracy: {best_score:.4f}")
    print(f"  Best Params: {best_config['Best Parameters']}")

print("\n===================================================")
print("ANN Simulation Completed Successfully")
print(f"Results saved to:\n{result_file}")
print("===================================================")
